# 🏭 07 — Filtro Industria 4.0 / Transizione 4.0

Carica tutti i file `classified_multiclass_aiuti_{year}.csv` (2014–2025) e produce un CSV unificato contenente i soli record i cui campi **Misura** (`TITOLO_MISURA`), **Titolo** (`TITOLO_PROGETTO`) o **Descrizione** (`DESCRIZIONE_PROGETTO`) menzionano:

- `Industria 4.0`
- `Transizione 4.0`
- `Impresa 4.0` (rebrand 2017)
- forme "Piano (Nazionale) Industria/Transizione/Impresa 4.0"

Il match generico `4.0` isolato è **escluso** per evitare falsi positivi (versioni software, release, ecc.).

**Output:** `data/export/industria_transizione_4_0.csv`

## 1. Setup

In [1]:
import re
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data')
OUT_PATH = DATA_DIR / 'export' / 'industria_transizione_4_0.csv'
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2014, 2026))
TEXT_COLS = ['TITOLO_MISURA', 'TITOLO_PROGETTO', 'DESCRIZIONE_PROGETTO']

print('Setup completato ✓')

Setup completato ✓


## 2. Regex

Pattern unico, case-insensitive, ancorato a word boundary. Tollera separatori `-`, `_`, `.` e spazi multipli tra la parola chiave e `4.0`, e tra `4` e `0`.

Esempi di match: `Industria 4.0`, `industria4.0`, `Industria-4.0`, `transizione 4 . 0`, `Piano Nazionale Industria 4.0`, `Piano Impresa 4.0`.

Esempi di non-match: `versione 4.0`, `release 4.0`, `ISO 4.0`.

In [2]:
PATTERN_STR = r"\b(?:piano\s+(?:nazionale\s+)?)?(?:industria|transizione|impresa)\s*[-_.]?\s*4[\s.]*0\b"
PATTERN = re.compile(PATTERN_STR, re.IGNORECASE)

# Sanity check rapido
_tests_positive = [
    'Piano Industria 4.0', 'industria 4.0', 'Industria4.0', 'INDUSTRIA-4.0',
    'Transizione 4.0', 'transizione4.0', 'Piano Nazionale Transizione 4.0',
    'Impresa 4.0', 'Piano Impresa 4.0', 'Credito imposta Transizione 4.0 beni strumentali',
]
_tests_negative = [
    'versione 4.0', 'release 4.0', 'software 4.0', 'Windows 4.0', 'ISO 9001:4.0',
    'industria alimentare', 'transizione ecologica',
]

pos_ok = all(PATTERN.search(s) for s in _tests_positive)
neg_ok = all(not PATTERN.search(s) for s in _tests_negative)
print(f'Positive match : {pos_ok}')
print(f'Negative reject: {neg_ok}')
assert pos_ok and neg_ok, 'Regex self-test fallito'

Positive match : True
Negative reject: True


## 3. Caricamento e filtro per anno

Caricamento chunked per contenere l'uso di memoria (~24M righe totali). Per ogni chunk si applica il filtro regex sui tre campi testuali e si conservano solo le righe che combaciano.

In [ ]:
def filter_year(year: int, chunksize: int = 100_000) -> pd.DataFrame:
    path = DATA_DIR / f'classified_multiclass_aiuti_{year}.csv'
    if not path.exists():
        print(f'  ⚠ File non trovato: {path.name}')
        return pd.DataFrame()

    kept = []
    for chunk in pd.read_csv(path, chunksize=chunksize, low_memory=False):
        combined = (
            chunk[TEXT_COLS]
            .fillna('')
            .astype(str)
            .agg(' || '.join, axis=1)
        )
        mask = combined.str.contains(PATTERN, regex=True, na=False)
        if mask.any():
            kept.append(chunk.loc[mask].copy())

    return pd.concat(kept, ignore_index=True) if kept else pd.DataFrame()


frames = []
for y in YEARS:
    print(f'Processing {y}...', end=' ')
    df_y = filter_year(y)
    print(f'{len(df_y):>8,} match')
    if not df_y.empty:
        frames.append(df_y)

df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
del frames
print(f'\n✅ Totale record filtrati: {len(df):,}')

Processing 2014...        0 match
Processing 2015...        0 match
Processing 2016...        0 match
Processing 2017...      177 match
Processing 2018...    2,096 match
Processing 2019...    1,958 match
Processing 2020... 

## 4. Diagnostica

Verifica dove avviene il match (quale dei tre campi) e le misure più frequenti per sanity check.

In [ ]:
if not df.empty:
    # Distribuzione per anno di concessione
    if 'ANNO' in df.columns:
        per_year = df.groupby('ANNO').size().rename('num_record')
        print('Record per anno:')
        print(per_year.to_string())
        print()

    # Match per campo
    print('Match per campo (un record può contribuire a più campi):')
    for col in TEXT_COLS:
        n = df[col].fillna('').astype(str).str.contains(PATTERN, regex=True, na=False).sum()
        print(f'  {col:<25} {n:>8,}')
    print()

    # Top 20 misure
    print('Top 20 TITOLO_MISURA:')
    print(df['TITOLO_MISURA'].value_counts().head(20).to_string())

## 5. Export

In [ ]:
if df.empty:
    print('⚠ Nessun record da esportare.')
else:
    df.to_csv(OUT_PATH, index=False)
    size_mb = OUT_PATH.stat().st_size / 1e6
    print(f'✅ Scritto: {OUT_PATH}')
    print(f'   Righe  : {len(df):,}')
    print(f'   Colonne: {len(df.columns)}')
    print(f'   Size   : {size_mb:,.2f} MB')